# Vector retrieval + evaluation — a diagnostic client of `engineering_rag`

**This notebook contains no retrieval or evaluation implementation.** It imports the production package (`engineering_rag.pipelines.retrieval_pipeline`, `engineering_rag.services.retriever`) and calls it exactly as `engrag-retrieve` does. See `docs/retrieval/ARCHITECTURE.md` and `docs/retrieval/EVALUATION.md` for the full contract.

In [ ]:
import json
import os

from engineering_rag.utils.paths import repo_root

ROOT = repo_root()
os.chdir(ROOT)  # so relative paths in YAML profiles resolve against the repo root, matching the CLI
print("Repository root:", ROOT)

## 1. Load the retrieval profile

In [ ]:
from engineering_rag.pipelines.retrieval_config import load_retrieval_config

config = load_retrieval_config("configs/retrieval_production.yaml")
print("Model:", config.embedding.model_name)
print("Collection:", config.chroma.collection_name, "at", config.chroma.persistence_path)
print("Default top_k:", config.search.default_top_k, " Maximum top_k:", config.search.maximum_top_k)

## 2. Inspect the existing Chroma collection (non-mutating)

In [ ]:
from engineering_rag.pipelines.retrieval_pipeline import inspect_collection

report = inspect_collection(config)
print("Exists:", report.exists, " Count:", report.count)
print("Distance metric:", report.distance_metric, " Embedding dimension:", report.embedding_dimension)
print("Source distribution:", report.source_filename_distribution)

## 3. Embed a real query and retrieve top-k evidence

This calls the exact same `run_search` the `engrag-retrieve search` CLI command calls — BGE query embedding (with the required query instruction, applied once) into the real 768-D cosine collection above.

In [ ]:
from engineering_rag.pipelines.retrieval_pipeline import run_search

response = run_search(
    "What is the purpose of front-end engineering design?", config, top_k=5
)
print(f"{response.returned_count} hit(s) in {response.total_duration_s * 1000:.1f} ms "
      f"(embed={response.embedding_duration_s * 1000:.1f}ms, db={response.database_duration_s * 1000:.1f}ms)")

In [ ]:
for hit in response.hits:
    print(f"#{hit.rank}  {hit.chunk_id}  similarity={hit.similarity_score:.4f}  distance={hit.raw_distance:.4f}")
    print(f"   source={hit.source_filename}  pages={hit.page_numbers}  section={hit.section_title!r}")
    print("   ", hit.retrieval_text[:160].replace("\n", " "), "...")
    print()

## 4. Provenance is complete, not just a snippet

Every hit carries the full stored provenance — heading path, prev/next chunk links, source element refs, content hash — not just the retrieved text.

In [ ]:
top = response.hits[0]
print(json.dumps(top.model_dump(mode="json"), indent=2, ensure_ascii=False))

## 5. Metadata-filtered retrieval

Filters only support scalar Chroma metadata fields (see `search.allowed_metadata_filter_fields` in the profile) — JSON-encoded list fields like `page_numbers` are explicitly rejected, not silently ignored.

In [ ]:
filtered = run_search(
    "instrument index", config, top_k=3,
    metadata_filters={"source_filename": "Instrumentation-and-Control-Engineering.pdf"},
)
for hit in filtered.hits:
    print(hit.rank, hit.chunk_id, hit.source_filename)

## 6. Run a small retrieval evaluation

This calls the same `run_evaluation_pipeline` the `engrag-retrieve evaluate` CLI command calls, against the full 20-case ground-truth dataset (`data/eval/retrieval_ground_truth.jsonl`) at K = 1, 3, 5, 10. See `docs/retrieval/EVALUATION.md` for the metrics definitions, the cosine distance -> similarity verification, and the human-review status of every label.

In [ ]:
from engineering_rag.pipelines.retrieval_pipeline import run_evaluation_pipeline

run_dir, rows, summary = run_evaluation_pipeline(
    config, reproduction_command="engrag-retrieve evaluate --profile configs/retrieval_production.yaml"
)
print("Run directory:", run_dir.root)
print(f"Cases: {summary.case_count}  (positive={summary.positive_case_count}, "
      f"negative={summary.negative_case_count})")
print(f"Human-reviewed: {summary.human_reviewed_count}/{summary.case_count}")

In [ ]:
for k in summary.k_values:
    print(f"K={k:<2}  HitRate={summary.hit_rate_at_k[k]:.3f}  Recall={summary.recall_at_k[k]:.3f}  "
          f"Precision={summary.precision_at_k[k]:.3f}  nDCG={summary.ndcg_at_k[k]:.3f}")
print(f"\nMRR: {summary.mean_reciprocal_rank:.3f}")
print(f"No-result accuracy (heuristic): {summary.no_result_accuracy}")

## 7. Honesty check: read the limitations before trusting the numbers above

In [ ]:
for item in summary.limitations:
    print("-", item)

## Next milestone

BM25/hybrid retrieval and reciprocal-rank fusion, evaluated against this same ground-truth dataset and metrics so any improvement is measured, not assumed. See `docs/retrieval/RETRIEVAL_COMPLETION_REPORT.md`.